# Tutorial: Embed Map Denoise

Audience:
- Python users who want derived views of a metric space.

Prerequisites:
- Callable metrics and simple Python records.

Learning goals:
- Embed a one-dimensional finite metric space.
- Map records into a derived metric space.
- Denoise by removing isolated records.

## Outline

1. Import the public facade.
2. Build a numeric process space.
3. Embed records into coordinates.
4. Map records into structured records.
5. Denoise a space.

In [ ]:
from metric import Space
from metric import metrics


## Step 1 - Build a numeric metric space

A plain absolute-distance callable is enough for this tiny process summary.

In [ ]:
records = [0, 1, 2, 10]
space = Space(records, metric=lambda lhs, rhs: abs(lhs - rhs))

assert space.distance(0, 3) == 10
space.pairwise()


## Step 2 - Embed

Embedding is still a native-only path in the default wheel; this notebook pins the explicit error.

In [ ]:
from metric.exceptions import StrategyUnavailableError

try:
    space.embed(dimensions=1)
    raise AssertionError("embed should require a native binding")
except StrategyUnavailableError as exc:
    message = str(exc)

assert "native C++ binding" in message
message


## Step 3 - Map into derived records

`map` preserves source lineage while changing record shape.

In [ ]:
mapped = space.map(
    transform=lambda value: {"value": value, "bucket": "high" if value >= 10 else "low"},
    metric=lambda lhs, rhs: abs(lhs["value"] - rhs["value"]),
)

assert mapped.source_record_ids == (0, 1, 2, 3)
assert mapped.space.records[0]["bucket"] == "low"
mapped.strategy


## Step 4 - Denoise

The default denoise path removes records selected by the native outlier score.

In [ ]:
clean = space.denoise(count=1)

assert clean.source_record_ids == (0, 1, 2)
assert clean.space.records == [0, 1, 2]
clean.to_dict()


## Exercise

Change `count=1` to `count=2` and inspect which source records remain.

Common pitfall: denoising is lossy. Use `result.source_record_ids` to keep lineage for downstream analysis.

In [ ]:
clean = space.denoise(count=2)

assert len(clean.space.records) == 2
clean.space.records
